# SC-6-Errors-Events - Erreurs et Evenements

[<< Inheritance](SC-5-Inheritance.ipynb) | [Token Standards >>](../02-Solidity-Advanced/SC-7-Token-Standards.ipynb)

---

## Objectifs d'apprentissage

1. Gerer les erreurs avec `require`, `revert`, `assert`
2. Creer des **custom errors** (Solidity 0.8.4+)
3. Emettre des **events** pour le logging

### Prerequis

- [SC-1](SC-3-Solidity-Basics.ipynb) a [SC-3](SC-5-Inheritance.ipynb) completes
- Comprendre les transactions Ethereum (gas, reverts)

### Duree estimee : 30 minutes

---

## 0. Connexion a la blockchain locale

Tous les contrats de ce notebook sont **compiles et deployes reellement** sur anvil.
Lancez `anvil` dans un terminal avant d'executer les cellules.


In [1]:
# Connection a anvil (blockchain locale Foundry)
# Prerequis: anvil en cours d'execution dans un terminal
from web3 import Web3
import solcx

SOLC_VERSION = "0.8.28"
ANVIL_URL = "http://127.0.0.1:8545"

# Connexion
w3 = Web3(Web3.HTTPProvider(ANVIL_URL))
assert w3.is_connected(), f"Impossible de se connecter a {ANVIL_URL}. Lancez 'anvil' dans un terminal."

# Installer solc si necessaire
installed = [str(v) for v in solcx.get_installed_solc_versions()]
if SOLC_VERSION not in installed:
    solcx.install_solc(SOLC_VERSION)
solcx.set_solc_version(SOLC_VERSION)

deployer = w3.eth.accounts[0]
print(f"Connecte a anvil (chain {w3.eth.chain_id}), deployer: {deployer[:10]}...")


def compile_and_deploy(w3, source_code, deployer, *constructor_args):
    """Compiler et deployer un contrat Solidity (source a un seul contrat)."""
    compiled = solcx.compile_source(
        source_code, output_values=["abi", "bin"], solc_version=SOLC_VERSION
    )
    contract_id, contract_interface = compiled.popitem()
    Contract = w3.eth.contract(
        abi=contract_interface["abi"], bytecode=contract_interface["bin"]
    )
    tx_hash = Contract.constructor(*constructor_args).transact({"from": deployer})
    receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
    instance = w3.eth.contract(
        address=receipt.contractAddress, abi=contract_interface["abi"]
    )
    print(f"Deploye: {contract_id.split(':')[-1]} a {receipt.contractAddress}")
    return instance, receipt


def deploy_named(w3, source_code, contract_name, deployer, *constructor_args):
    """Compiler une source multi-contrats et deployer un contrat specifique par nom."""
    compiled = solcx.compile_source(
        source_code, output_values=["abi", "bin"], solc_version=SOLC_VERSION
    )
    for contract_id, contract_interface in compiled.items():
        if contract_id.split(':')[-1] == contract_name:
            Contract = w3.eth.contract(
                abi=contract_interface["abi"], bytecode=contract_interface["bin"]
            )
            tx_hash = Contract.constructor(*constructor_args).transact({"from": deployer})
            receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
            instance = w3.eth.contract(
                address=receipt.contractAddress, abi=contract_interface["abi"]
            )
            print(f"Deploye: {contract_name} a {receipt.contractAddress}")
            return instance, receipt
    raise ValueError(f"Contrat '{contract_name}' introuvable dans la source compilee")

Connecte a anvil (chain 31337), deployer: 0xf39Fd6e5...


---

## 1. Gestion des erreurs

| Fonction | Usage | Gas remboursé |
|----------|-------|---------------|
| `require` | Conditions d'entree | Oui |
| `revert` | Erreurs complexes | Oui |
| `assert` | Invariants internes | Non |

In [2]:
# Gestion des erreurs
ERROR_HANDLING = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract ErrorHandling {
    address public owner;
    uint256 public value;

    constructor() {
        owner = msg.sender;
    }

    function setValue(uint256 _value) public {
        require(msg.sender == owner, "Not owner");
        require(_value > 0, "Value must be positive");
        value = _value;
    }

    function complexCheck(uint256 a, uint256 b) public pure {
        if (a == 0) { revert("A cannot be zero"); }
        if (b == 0) { revert("B cannot be zero"); }
        if (a + b > 100) { revert("Sum too large"); }
    }

    function invariant() public view {
        assert(owner != address(0));
    }
}
'''

print("--- require / revert / assert ---")
eh, _ = compile_and_deploy(w3, ERROR_HANDLING, deployer)
print(f"  Deploye : {eh.address}")

# require : not owner
other = w3.eth.accounts[1]
try:
    eh.functions.setValue(42).transact({'from': other})
except Exception:
    print("  setValue(42) par other → revert 'Not owner' (attendu)")

# require : value must be positive
try:
    eh.functions.setValue(0).transact({'from': deployer})
except Exception:
    print("  setValue(0) → revert 'Value must be positive' (attendu)")

# require : success
eh.functions.setValue(42).transact({'from': deployer})
print(f"  setValue(42) par owner → value = {eh.functions.value().call()}")

# revert : sum too large
try:
    eh.functions.complexCheck(60, 50).transact({'from': deployer})
except Exception:
    print("  complexCheck(60, 50) → revert 'Sum too large' (attendu)")

# assert : invariant holds
eh.functions.invariant().call()
print("  invariant() → passe (owner != address(0))")

--- require / revert / assert ---


Deploye: ErrorHandling a 0x5FbDB2315678afecb367f032d93F642f64180aa3
  Deploye : 0x5FbDB2315678afecb367f032d93F642f64180aa3


  setValue(42) par other → revert 'Not owner' (attendu)


  setValue(0) → revert 'Value must be positive' (attendu)


  setValue(42) par owner → value = 42


  complexCheck(60, 50) → revert 'Sum too large' (attendu)
  invariant() → passe (owner != address(0))


---

## 2. Custom Errors (Solidity 0.8.4+)

Les custom errors sont plus economiques en gas que les strings.

In [3]:
# Custom errors
CUSTOM_ERRORS = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

error InsufficientBalance(uint256 available, uint256 required);
error Unauthorized(address caller);
error InvalidAmount();

contract CustomErrors {
    mapping(address => uint256) public balances;

    function deposit() public payable {
        balances[msg.sender] += msg.value;
    }

    function withdraw(uint256 amount) public {
        if (balances[msg.sender] < amount) {
            revert InsufficientBalance(balances[msg.sender], amount);
        }
        balances[msg.sender] -= amount;
        payable(msg.sender).transfer(amount);
    }

    function transfer(address to, uint256 amount) public {
        if (amount == 0) {
            revert InvalidAmount();
        }
        if (balances[msg.sender] < amount) {
            revert InsufficientBalance(balances[msg.sender], amount);
        }
        balances[msg.sender] -= amount;
        balances[to] += amount;
    }
}
'''

print("--- Custom errors avec parametres ---")
ce, _ = compile_and_deploy(w3, CUSTOM_ERRORS, deployer)
receiver = w3.eth.accounts[1]

# Deposit 0.5 ETH
ce.functions.deposit().transact({'from': deployer, 'value': w3.to_wei(0.5, 'ether')})
print(f"  Apres deposit(0.5 ETH) : balance = {w3.from_wei(ce.functions.balances(deployer).call(), 'ether')} ETH")

# withdraw trop → InsufficientBalance
try:
    ce.functions.withdraw(w3.to_wei(1, 'ether')).transact({'from': deployer})
except Exception as e:
    print(f"  withdraw(1 ETH) → revert InsufficientBalance (attendu)")

# transfer(0) → InvalidAmount
try:
    ce.functions.transfer(receiver, 0).transact({'from': deployer})
except Exception:
    print("  transfer(0) → revert InvalidAmount (attendu)")

# transfer valide
ce.functions.transfer(receiver, w3.to_wei(0.1, 'ether')).transact({'from': deployer})
print(f"  Apres transfer(0.1 ETH) : receiver balance = {w3.from_wei(ce.functions.balances(receiver).call(), 'ether')} ETH")

--- Custom errors avec parametres ---


Deploye: CustomErrors a 0x9fE46736679d2D9a65F0992F2272dE9f3c7fa6e0


  Apres deposit(0.5 ETH) : balance = 0.5 ETH


  withdraw(1 ETH) → revert InsufficientBalance (attendu)


  transfer(0) → revert InvalidAmount (attendu)


  Apres transfer(0.1 ETH) : receiver balance = 0.1 ETH


---

## 3. Events

Les events permettent de logger des donnees sur la blockchain.

In [4]:
# Events
EVENTS_EXAMPLE = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract EventsExample {
    event Deposit(address indexed from, uint256 amount, uint256 timestamp);
    event Withdrawal(address indexed to, uint256 amount);
    event Transfer(address indexed from, address indexed to, uint256 amount);
    event OwnershipTransferred(address indexed previousOwner, address indexed newOwner);

    address public owner;
    mapping(address => uint256) public balances;

    constructor() {
        owner = msg.sender;
    }

    function deposit() public payable {
        balances[msg.sender] += msg.value;
        emit Deposit(msg.sender, msg.value, block.timestamp);
    }

    function withdraw(uint256 amount) public {
        require(balances[msg.sender] >= amount, "Insufficient balance");
        balances[msg.sender] -= amount;
        payable(msg.sender).transfer(amount);
        emit Withdrawal(msg.sender, amount);
    }

    function transferFunds(address to, uint256 amount) public {
        require(balances[msg.sender] >= amount, "Insufficient balance");
        balances[msg.sender] -= amount;
        balances[to] += amount;
        emit Transfer(msg.sender, to, amount);
    }

    function transferOwnership(address newOwner) public {
        require(msg.sender == owner, "Not owner");
        require(newOwner != address(0), "Invalid address");
        address previousOwner = owner;
        owner = newOwner;
        emit OwnershipTransferred(previousOwner, newOwner);
    }
}
'''

print("--- Events : Deposit, Transfer, OwnershipTransferred ---")
ev, _ = compile_and_deploy(w3, EVENTS_EXAMPLE, deployer)
receiver = w3.eth.accounts[1]

# Deposit → event Deposit
tx = ev.functions.deposit().transact({'from': deployer, 'value': w3.to_wei(1, 'ether')})
receipt = w3.eth.get_transaction_receipt(tx)
logs = ev.events.Deposit().process_receipt(receipt)
print(f"  Deposit event : from={logs[0]['args']['from'][:10]}..., amount={w3.from_wei(logs[0]['args']['amount'], 'ether')} ETH")

# transferFunds → event Transfer
tx = ev.functions.transferFunds(receiver, w3.to_wei(0.3, 'ether')).transact({'from': deployer})
receipt = w3.eth.get_transaction_receipt(tx)
logs = ev.events.Transfer().process_receipt(receipt)
print(f"  Transfer event : from={logs[0]['args']['from'][:10]}..., to={logs[0]['args']['to'][:10]}..., amount={w3.from_wei(logs[0]['args']['amount'], 'ether')} ETH")

# transferOwnership → event OwnershipTransferred
new_owner = w3.eth.accounts[2]
tx = ev.functions.transferOwnership(new_owner).transact({'from': deployer})
receipt = w3.eth.get_transaction_receipt(tx)
logs = ev.events.OwnershipTransferred().process_receipt(receipt)
print(f"  OwnershipTransferred : previous={logs[0]['args']['previousOwner'][:10]}..., new={logs[0]['args']['newOwner'][:10]}...")

--- Events : Deposit, Transfer, OwnershipTransferred ---


Deploye: EventsExample a 0x5FC8d32690cc91D4c39d9d3abcBD16989F875707


  Deposit event : from=0xf39Fd6e5..., amount=1 ETH


  Transfer event : from=0xf39Fd6e5..., to=0x70997970..., amount=0.3 ETH


  OwnershipTransferred : previous=0xf39Fd6e5..., new=0x3C44CdDd...


### 3.1 Indexed parameters

- Maximum **3 parametres indexes** par event
- Les parametres indexes sont recherchables dans les logs
- Les parametres non indexes sont stockes dans `data`

In [5]:
# Indexed parameters
INDEXED_EXAMPLE = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract IndexedExample {
    event OrderCreated(
        uint256 indexed orderId,
        address indexed buyer,
        address indexed seller,
        uint256 amount,
        string productId
    );

    function createOrder(
        address seller,
        uint256 amount,
        string memory productId
    ) public returns (uint256) {
        uint256 orderId = uint256(keccak256(abi.encodePacked(
            msg.sender, seller, amount, block.timestamp
        )));
        emit OrderCreated(orderId, msg.sender, seller, amount, productId);
        return orderId;
    }
}
'''

print("--- Indexed parameters : recherchables dans les logs ---")
idx, _ = compile_and_deploy(w3, INDEXED_EXAMPLE, deployer)
seller = w3.eth.accounts[1]

tx = idx.functions.createOrder(seller, 500, "PROD-001").transact({'from': deployer})
receipt = w3.eth.get_transaction_receipt(tx)
logs = idx.events.OrderCreated().process_receipt(receipt)
args = logs[0]['args']
print(f"  orderId (indexed) = {hex(args['orderId'])[:18]}...")
print(f"  buyer   (indexed) = {args['buyer'][:10]}...")
print(f"  seller  (indexed) = {args['seller'][:10]}...")
print(f"  amount  (data)    = {args['amount']}")
print(f"  productId (data)  = '{args['productId']}'")
print("  Les 3 premiers champs sont indexables par filtrage, les 2 derniers sont dans data")

--- Indexed parameters : recherchables dans les logs ---


Deploye: IndexedExample a 0x8A791620dd6260079BF849Dc5567aDC3F2FdC318


  orderId (indexed) = 0xb6cda92322db49d9...
  buyer   (indexed) = 0xf39Fd6e5...
  seller  (indexed) = 0x70997970...
  amount  (data)    = 500
  productId (data)  = 'PROD-001'
  Les 3 premiers champs sont indexables par filtrage, les 2 derniers sont dans data


---

## 4. Try/Catch pour appels externes

In [6]:
# Try/Catch
TRY_CATCH_SOURCE = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

// Contrat externe controllable pour la demo
contract MockExternal {
    bool public shouldFail = true;

    function riskyOperation() external returns (bool) {
        require(!shouldFail, "Operation failed by design");
        return true;
    }

    function setFail(bool _fail) external {
        shouldFail = _fail;
    }
}

interface IExternalContract {
    function riskyOperation() external returns (bool);
}

contract TryCatchExample {
    event OperationSuccess(bool result);
    event OperationFailed(string reason);

    IExternalContract public externalContract;

    constructor(address _contract) {
        externalContract = IExternalContract(_contract);
    }

    function safeOperation() public returns (bool) {
        try externalContract.riskyOperation() returns (bool success) {
            emit OperationSuccess(success);
            return success;
        } catch Error(string memory reason) {
            emit OperationFailed(reason);
            return false;
        } catch {
            emit OperationFailed("Unknown error");
            return false;
        }
    }
}
'''

print("--- Try/Catch pour appels externes ---")

# Deployer MockExternal en premier
mock, _ = deploy_named(w3, TRY_CATCH_SOURCE, "MockExternal", deployer)

# Deployer TryCatchExample en pointant vers MockExternal
tce, _ = deploy_named(w3, TRY_CATCH_SOURCE, "TryCatchExample", deployer, mock.address)
print(f"  MockExternal : shouldFail = {mock.functions.shouldFail().call()}")

# safeOperation() quand shouldFail=True → catch Error
tx = tce.functions.safeOperation().transact({'from': deployer})
receipt = w3.eth.get_transaction_receipt(tx)
failed_logs = tce.events.OperationFailed().process_receipt(receipt)
print(f"  safeOperation() [shouldFail=True] → OperationFailed: '{failed_logs[0]['args']['reason']}'")

# Changer shouldFail a False
mock.functions.setFail(False).transact({'from': deployer})

# safeOperation() quand shouldFail=False → success
tx = tce.functions.safeOperation().transact({'from': deployer})
receipt = w3.eth.get_transaction_receipt(tx)
success_logs = tce.events.OperationSuccess().process_receipt(receipt)
print(f"  safeOperation() [shouldFail=False] → OperationSuccess: {success_logs[0]['args']['result']}")

--- Try/Catch pour appels externes ---


Deploye: MockExternal a 0xB7f8BC63BbcaD18155201308C8f3540b07f84F5e


Deploye: TryCatchExample a 0xA51c1fc2f0D1a1b8494Ed1FE312d7C3a78Ed91C0
  MockExternal : shouldFail = True


  safeOperation() [shouldFail=True] → OperationFailed: 'Operation failed by design'


  safeOperation() [shouldFail=False] → OperationSuccess: True


### Exercice : Gestionnaire de whitelist avec errors et events

Creez un contrat `WhitelistManager` qui gere une liste d'adresses autorisees, utilisant des custom errors pour les cas d'erreur et des events pour tracer les modifications.

**Indices** :
- Definissez 3 custom errors : `NotOwner()`, `AlreadyWhitelisted(address)`, `NotWhitelisted(address)`
- Definissez 2 events : `Whitelisted(address indexed account)`, `RemovedFromWhitelist(address indexed account)`
- Utilisez un `mapping(address => bool)` pour suivre les adresses
- Utilisez un `address public owner` avec un modificateur `onlyOwner`

In [7]:
# Exercice : Gestionnaire de whitelist avec errors et events
EXERCICE_WHITELIST = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

// TODO etudiant : definir les 3 custom errors (NotOwner, AlreadyWhitelisted, NotWhitelisted)

contract WhitelistManager {
    // TODO etudiant : definir owner, mapping whitelist, 2 events
    
    constructor() {
        // TODO etudiant : initialiser owner
        pass
    }
    
    modifier onlyOwner() {
        // TODO etudiant : verifier msg.sender == owner, sinon revert NotOwner()
        _;
    }
    
    function addToWhitelist(address account) public onlyOwner {
        // TODO etudiant : verifier que account n'est pas deja whitelist (AlreadyWhitelisted)
        // TODO etudiant : ajouter au mapping et emettre l'event Whitelisted
        pass
    }
    
    function removeFromWhitelist(address account) public onlyOwner {
        // TODO etudiant : verifier que account est whitelist (NotWhitelisted)
        // TODO etudiant : retirer du mapping et emettre l'event RemovedFromWhitelist
        pass
    }
    
    function isWhitelisted(address account) public view returns (bool) {
        // TODO etudiant : retourner le statut
        return false;
    }
}
'''

print("Exercice a completer : WhitelistManager avec custom errors et events")

Exercice a completer : WhitelistManager avec custom errors et events


### Exercice 1 : Controle d'acces par events

Creez un contrat `EventAccessControl` qui utilise des events comme audit trail pour un systeme de roles (admin, editor, viewer). Chaque changement de role doit emettre un event, et les custom errors doivent signaler les violations d'acces.

**Objectifs** :
1. Implementer un systeme de roles avec 3 niveaux (ADMIN, EDITOR, VIEWER)
2. Emettre des events pour chaque changement de role (attribution, retrait)
3. Utiliser des custom errors pour les acces non autorises

**Indices** :
- Definissez les errors : `UnauthorizedRole(address caller, bytes32 requiredRole)`, `RoleAlreadyGranted(address account, bytes32 role)`
- Definissez les events : `RoleGranted(address indexed account, bytes32 indexed role, address indexed grantedBy)`, `RoleRevoked(address indexed account, bytes32 indexed role, address indexed revokedBy)`
- Utilisez un `mapping(address => bytes32)` pour stocker le role de chaque adresse
- Representez les roles par des `bytes32` constants (ex: `keccak256("ADMIN")`)

In [8]:
# Exercice 1 : Controle d'acces par events
EXERCICE_ACCESS_CONTROL = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

// TODO etudiant : definir custom errors UnauthorizedRole et RoleAlreadyGranted

contract EventAccessControl {
    // TODO etudiant : definir bytes32 constants ADMIN, EDITOR, VIEWER
    // TODO etudiant : definir mapping(address => bytes32) roles
    // TODO etudiant : definir events RoleGranted et RoleRevoked

    address public owner;

    constructor() {
        // TODO etudiant : initialiser owner et lui attribuer le role ADMIN
        pass
    }

    modifier onlyRole(bytes32 _role) {
        // TODO etudiant : verifier que msg.sender a le role requis
        _;
    }

    function grantRole(address account, bytes32 role) public onlyRole(ADMIN) {
        // TODO etudiant : verifier que le role n'est pas deja attribue (RoleAlreadyGranted)
        // TODO etudiant : attribuer le role et emettre RoleGranted
        pass
    }

    function revokeRole(address account) public onlyRole(ADMIN) {
        // TODO etudiant : retirer le role et emettre RoleRevoked
        pass
    }

    function hasRole(address account, bytes32 role) public view returns (bool) {
        // TODO etudiant : verifier si l'adresse a le role donne
        return false;
    }
}
'''

print("Exercice a completer : EventAccessControl avec roles et audit trail")

Exercice a completer : EventAccessControl avec roles et audit trail


---

## 5. Exercices

### Exercice 2 : Contrat avec custom errors

Creez un contrat de vote avec custom errors pour les cas invalides.

In [9]:
# Exercice 2 : Contrat de vote avec custom errors
EXERCICE_VOTING = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

// Custom errors
error AlreadyVoted(address voter);
error VotingClosed();
error InvalidProposal(uint256 proposalId);

contract Voting {
    struct Proposal {
        string name;
        uint256 voteCount;
    }

    Proposal[] public proposals;
    mapping(address => bool) public hasVoted;
    bool public votingOpen = true;

    event Voted(address indexed voter, uint256 indexed proposalId);
    event VotingFinalized(uint256 timestamp);

    constructor(string[] memory _proposalNames) {
        for (uint i = 0; i < _proposalNames.length; i++) {
            proposals.push(Proposal(_proposalNames[i], 0));
        }
    }

    function vote(uint256 proposalId) public {
        if (!votingOpen) revert VotingClosed();
        if (hasVoted[msg.sender]) revert AlreadyVoted(msg.sender);
        if (proposalId >= proposals.length) revert InvalidProposal(proposalId);
        hasVoted[msg.sender] = true;
        proposals[proposalId].voteCount++;
        emit Voted(msg.sender, proposalId);
    }

    function closeVoting() public {
        votingOpen = false;
        emit VotingFinalized(block.timestamp);
    }
}
'''

print("--- Exercice 2 : Voting avec custom errors ---")
voting, _ = compile_and_deploy(w3, EXERCICE_VOTING, deployer, ["Alice", "Bob", "Charlie"])

voter1 = w3.eth.accounts[1]
voter2 = w3.eth.accounts[2]
voter3 = w3.eth.accounts[3]

# Votes valides
for voter, proposal_id in [(voter1, 0), (voter2, 1), (voter3, 0)]:
    tx = voting.functions.vote(proposal_id).transact({'from': voter})
    receipt = w3.eth.get_transaction_receipt(tx)
    logs = voting.events.Voted().process_receipt(receipt)
    print(f"  {voter[:10]}... a vote pour proposalId={logs[0]['args']['proposalId']}")

# Verifier les compteurs
for i in range(3):
    p = voting.functions.proposals(i).call()
    print(f"  Proposal {i} '{p[0]}' : {p[1]} vote(s)")

# AlreadyVoted
try:
    voting.functions.vote(0).transact({'from': voter1})
except Exception:
    print("  vote() depuis voter1 → revert AlreadyVoted (attendu)")

# InvalidProposal
try:
    voting.functions.vote(99).transact({'from': w3.eth.accounts[4]})
except Exception:
    print("  vote(99) → revert InvalidProposal (attendu)")

# Fermer le vote et emettre VotingFinalized
tx = voting.functions.closeVoting().transact({'from': deployer})
receipt = w3.eth.get_transaction_receipt(tx)
logs = voting.events.VotingFinalized().process_receipt(receipt)
print(f"  closeVoting() → VotingFinalized timestamp={logs[0]['args']['timestamp']}")

# VotingClosed apres fermeture
try:
    voting.functions.vote(1).transact({'from': w3.eth.accounts[5]})
except Exception:
    print("  vote() apres fermeture → revert VotingClosed (attendu)")

--- Exercice 2 : Voting avec custom errors ---


Deploye: Voting a 0x959922bE3CAee4b8Cd9a407cc3ac1C251C2007B1


  0x70997970... a vote pour proposalId=0


  0x3C44CdDd... a vote pour proposalId=1


  0x90F79bf6... a vote pour proposalId=0
  Proposal 0 'Alice' : 2 vote(s)


  Proposal 1 'Bob' : 1 vote(s)
  Proposal 2 'Charlie' : 0 vote(s)


  vote() depuis voter1 → revert AlreadyVoted (attendu)


  vote(99) → revert InvalidProposal (attendu)


  closeVoting() → VotingFinalized timestamp=1780879746


  vote() apres fermeture → revert VotingClosed (attendu)


### Exercice 3 : Contrat Auction avec events

Creez un contrat d'enchere avec events pour chaque action.

In [10]:
# Exercice 3 : Contrat Auction avec events
EXERCICE_AUCTION = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

error AuctionEnded();
error BidTooLow(uint256 currentBid, uint256 minBid);

contract Auction {
    address public beneficiary;
    uint256 public auctionEndTime;
    address public highestBidder;
    uint256 public highestBid;
    bool public ended;

    mapping(address => uint256) public pendingReturns;

    event HighestBidIncreased(address indexed bidder, uint256 amount);
    event AuctionFinalized(address winner, uint256 amount);
    event BidRefunded(address indexed bidder, uint256 amount);

    constructor(uint256 biddingTime, address _beneficiary) {
        beneficiary = _beneficiary;
        auctionEndTime = block.timestamp + biddingTime;
    }

    function bid() public payable {
        if (block.timestamp >= auctionEndTime || ended) revert AuctionEnded();
        if (msg.value <= highestBid) revert BidTooLow(highestBid, msg.value);
        if (highestBid > 0) {
            pendingReturns[highestBidder] += highestBid;
        }
        highestBidder = msg.sender;
        highestBid = msg.value;
        emit HighestBidIncreased(msg.sender, msg.value);
    }

    function withdraw() public returns (bool) {
        uint256 amount = pendingReturns[msg.sender];
        if (amount > 0) {
            pendingReturns[msg.sender] = 0;
            payable(msg.sender).transfer(amount);
            emit BidRefunded(msg.sender, amount);
        }
        return true;
    }

    function auctionEnd() public {
        require(block.timestamp >= auctionEndTime, "Auction not yet ended");
        require(!ended, "Auction already finalized");
        ended = true;
        emit AuctionFinalized(highestBidder, highestBid);
        payable(beneficiary).transfer(highestBid);
    }
}
'''

print("--- Exercice 3 : Auction avec events ---")
beneficiary_addr = w3.eth.accounts[3]
bidder1 = w3.eth.accounts[1]
bidder2 = w3.eth.accounts[2]

auction, _ = compile_and_deploy(w3, EXERCICE_AUCTION, deployer, 60, beneficiary_addr)

# bidder1 offre 1 ETH
tx = auction.functions.bid().transact({'from': bidder1, 'value': w3.to_wei(1, 'ether')})
receipt = w3.eth.get_transaction_receipt(tx)
logs = auction.events.HighestBidIncreased().process_receipt(receipt)
print(f"  bidder1 offre 1 ETH → HighestBidIncreased: {w3.from_wei(logs[0]['args']['amount'], 'ether')} ETH")

# bidder2 surencherit a 2 ETH → bidder1 entre dans pendingReturns
tx = auction.functions.bid().transact({'from': bidder2, 'value': w3.to_wei(2, 'ether')})
receipt = w3.eth.get_transaction_receipt(tx)
logs = auction.events.HighestBidIncreased().process_receipt(receipt)
print(f"  bidder2 offre 2 ETH → HighestBidIncreased: {w3.from_wei(logs[0]['args']['amount'], 'ether')} ETH")
print(f"  pendingReturns[bidder1] = {w3.from_wei(auction.functions.pendingReturns(bidder1).call(), 'ether')} ETH")

# BidTooLow
try:
    auction.functions.bid().transact({'from': w3.eth.accounts[4], 'value': w3.to_wei(0.5, 'ether')})
except Exception:
    print("  bid(0.5 ETH) → revert BidTooLow (attendu)")

# bidder1 recupere son ETH via withdraw()
tx = auction.functions.withdraw().transact({'from': bidder1})
receipt = w3.eth.get_transaction_receipt(tx)
logs = auction.events.BidRefunded().process_receipt(receipt)
print(f"  bidder1 withdraw() → BidRefunded: {w3.from_wei(logs[0]['args']['amount'], 'ether')} ETH")

# Avancer le temps de 61 secondes sur anvil pour expirer l'enchere
w3.provider.make_request("evm_increaseTime", [61])
w3.provider.make_request("evm_mine", [])

# auctionEnd() : transfère les fonds au beneficiaire
beneficiary_before = w3.eth.get_balance(beneficiary_addr)
tx = auction.functions.auctionEnd().transact({'from': deployer})
receipt = w3.eth.get_transaction_receipt(tx)
logs = auction.events.AuctionFinalized().process_receipt(receipt)
beneficiary_after = w3.eth.get_balance(beneficiary_addr)
gain = w3.from_wei(beneficiary_after - beneficiary_before, 'ether')
print(f"  auctionEnd() → AuctionFinalized: winner={logs[0]['args']['winner'][:10]}..., amount={w3.from_wei(logs[0]['args']['amount'], 'ether')} ETH")
print(f"  beneficiaire a recu +{gain} ETH")

# Double appel → revert Auction already finalized
try:
    auction.functions.auctionEnd().transact({'from': deployer})
except Exception:
    print("  auctionEnd() une seconde fois → revert 'Auction already finalized' (attendu)")

--- Exercice 3 : Auction avec events ---


Deploye: Auction a 0x68B1D87F95878fE05B998F19b66F4baba5De1aed


  bidder1 offre 1 ETH → HighestBidIncreased: 1 ETH


  bidder2 offre 2 ETH → HighestBidIncreased: 2 ETH
  pendingReturns[bidder1] = 1 ETH


  bid(0.5 ETH) → revert BidTooLow (attendu)


  bidder1 withdraw() → BidRefunded: 1 ETH


  auctionEnd() → AuctionFinalized: winner=0x3C44CdDd..., amount=2 ETH
  beneficiaire a recu +2 ETH


  auctionEnd() une seconde fois → revert 'Auction already finalized' (attendu)


### Exercice 4 : Hierarchie d'erreurs personnalisees

Creez un contrat `PaymentProcessor` qui gere des paiements avec une hierarchie de custom errors imbriquees. Certaines erreurs heritent d'autres contextes (erreur parent + erreur enfant), et le contrat doit les differencier dans ses reverts.

**Objectifs** :
1. Definir une hierarchie de custom errors avec des parametres
2. Implementer un contrat qui utilise differentes erreurs selon le contexte
3. Utiliser des events pour tracer les operations reussies et echouees

**Indices** :
- Definissez `PaymentError(address payer, uint256 amount)` comme erreur generique
- Definissez `InsufficientFunds(address payer, uint256 available, uint256 required)` et `PaymentExpired(uint256 deadline)` comme erreurs specifiques
- Utilisez un `mapping(address => uint256)` pour les soldes
- Emettez un event `PaymentProcessed(address indexed from, address indexed to, uint256 amount)` en cas de succes

In [11]:
# Exercice 4 : Hierarchie d'erreurs personnalisees
EXERCICE_PAYMENT_PROCESSOR = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

// TODO etudiant : definir les 3 custom errors (PaymentError, InsufficientFunds, PaymentExpired)

contract PaymentProcessor {
    // TODO etudiant : definir mapping balances, uint256 deadline, event PaymentProcessed

    constructor(uint256 _deadline) {
        // TODO etudiant : initialiser deadline (timestamp de validite)
        pass
    }

    function deposit() public payable {
        // TODO etudiant : incrementer le solde de msg.sender
        pass
    }

    function pay(address to, uint256 amount) public {
        // TODO etudiant : verifier deadline non depassee (PaymentExpired)
        // TODO etudiant : verifier solde suffisant (InsufficientFunds)
        // TODO etudiant : transferer et emettre PaymentProcessed
        pass
    }

    function getBalance(address account) public view returns (uint256) {
        // TODO etudiant : retourner le solde
        return 0;
    }
}
'''

print("Exercice a completer : PaymentProcessor avec hierarchie d'erreurs et events")

Exercice a completer : PaymentProcessor avec hierarchie d'erreurs et events


---

## 6. Resume

| Concept | Description | Gas |
|---------|-------------|-----|
| `require` | Conditions d'entree | Rembourse |
| `revert` | Erreurs complexes | Rembourse |
| `assert` | Invariants internes | Non rembourse |
| `error` | Custom errors (0.8.4+) | Economique |
| `event` | Logging blockchain | - |
| `indexed` | Parametres recherchables | Max 3 |

---

**Notebook suivant** : [SC-7-Token-Standards](../02-Solidity-Advanced/SC-7-Token-Standards.ipynb)

---

[<< Inheritance](SC-5-Inheritance.ipynb) | [Token Standards >>](../02-Solidity-Advanced/SC-7-Token-Standards.ipynb)